# Tutorial 1: Data Preparation

This tutorial covers loading, preprocessing, and saving multimodal data for deepSCENIC.

## Setup

First, let's import the required packages:

In [2]:
import anndata as ad
import scanpy as sc

import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

deepSCENIC version: 0.1.0


## Example Dataset: MM_lines (Melanoma Cell Lines)

This tutorial uses the MM_lines dataset from the [SCENIC+ paper](https://www.nature.com/articles/s41592-023-01938-4) {cite:p}`bravo2023scenicplus`. This dataset contains scRNA-seq and scATAC-seq data from nine human melanoma cell lines representing different melanoma states (melanocytic, mesenchymal, and intermediate).

### Data Availability

The raw data is publicly available from multiple sources:

| Source | Description | Link |
|--------|-------------|------|
| **GEO** | pre-processed gene expression and region accessibility matrices | [GSE210745](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE210745) |
| **SCope** | Pre-processed .loom files | [SCENIC+ Scope session](https://scope.aertslab.org/#/scenic-v2) |
| **UCSC Cell Browser** | Interactive explorer | [SCENIC+ UCSC session](https://genome-euro.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr3%3A69527724%2D69971905&hgsid=420343291_foyulRvnMasQg5MiIL1Hk6cbsQpn) |

To follow along with the tutorial we suggest to download and process following files from GEO:
- [Gene expression matrix](https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE210745&format=file&file=GSE210745%5Fmm%5Flines%5Fgene%5Fexpression%5Fmatrix%2Etsv%2Egz)
- [Chromatin accessibility matrix](https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE210745&format=file&file=GSE210745%5Fmm%5Flines%5Fregion%5Faccessibility%5Fmatrix%2Etsv%2Egz)
- [Cell metadata](https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE210745&format=file&file=GSE210745%5Fmm%5Flines%5Fcell%5Fmetadata%2Etsv%2Egz)


In [4]:
import os

data_dir = "/data/groups/vib.ai/stein.aerts/gpartel/software/deepSCENIC_analyses/data/MM_lines/"
assert os.path.exists(data_dir), "dir not found"

## scRNA and scATAC Preprocessing

deepSCENIC combines RNA expression data, chromatin accessibility data, and genomic sequences to learn gene regulatory networks at the single-cell level. deepSCENIC follows the `scverse` convention for data formatting, and therefore we expect that your data is already preprocessed into the `anndata.AnnData` format. We will further preprocess these data modalities separately and later combine them into one `mudata.MuData` object, which is the format that deepSCENIC uses to run both training and inference.

### scRNA-seq Preprocessing

Load the gene expression matrix:

In [5]:
# Load scRNA-seq data
adata_rna = ad.read_h5ad(f"{data_dir}/ad_rna_raw.h5ad")
print(f"RNA data: {adata_rna.n_obs} cells x {adata_rna.n_vars} genes")

RNA data: 936 cells x 17085 genes


We perform some basic filtering and normalization using standard `scanpy` functionality.

In [6]:
# Standard scRNA-seq preprocessing with scanpy
# (deepSCENIC expects log-normalized RNA data)
sc.pp.filter_genes(adata_rna, min_cells=10)
sc.pp.normalize_total(adata_rna)
sc.pp.log1p(adata_rna)

Additionally, we remove zero-variance genes (if there are any). These are genes that have the same constant expression across all cells (e.g. 0), which means they'll likely be completely uninformative for training.

In [7]:
ds.pp.remove_zero_variance_genes(adata_rna)

In [8]:
adata_rna

AnnData object with n_obs × n_vars = 936 × 15684
    var: 'n_cells'
    uns: 'log1p'

#### Fetch External Resources

Before further preprocessing, we need to fetch transcription factor lists and gene annotations from external databases.

In [9]:
# Fetch TF list from SCENIC+ resources
tfs = ds.fetch_tf_collection(species="human")  # also available: "mouse", "fly"
print(f"Found {len(tfs)} transcription factors")
tfs[:10]

Found 1892 transcription factors


['ZNF354C',
 'KLF12',
 'ZNF143',
 'ZIC2',
 'ZNF274',
 'SP2',
 'ZBTB7A',
 'BCL6B',
 'ZBTB49',
 'ZIC1']

In [10]:
# Fetch gene annotations from Ensembl
annot, chromsizes = ds.fetch_gene_annotation(
    species="hsapiens",
    biomart_host="http://www.ensembl.org",  # default
    transcript_type="protein_coding",
)
print(f"Annotations for {len(annot)} genes")
annot.head()

deepscenic.datasets - INFO - Fetching gene annotation from http://www.ensembl.org...
deepscenic.datasets - INFO - Connecting to Biomart host: http://www.ensembl.org
deepscenic.datasets - INFO - Querying gene annotation for hsapiens...
/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/pybiomart/dataset.py:269: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  result = pd.read_csv(StringIO(response.text), sep='\t')
deepscenic.datasets - INFO - Using genome assembly: GRCh38.p14
deepscenic.datasets - INFO - Found NCBI assembly ID: 11968211
deepscenic.datasets - INFO - Downloading assembly report from: ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/GCF_000001405.40_GRCh38.p14/GCF_000001405.40_GRCh38.p14_assembly_report.txt
deepscenic.datasets - INFO - Found 25 assembled chromosomes
deepscenic.datasets - INFO - Converted chromosome names to UCSC style
deepscenic.datasets - INFO 

Annotations for 19357 genes


,Chromosome,Start,End,Strand,Transcription_Start_Site,Transcript_type
Gene,,,,,,
MT-ND1,chrM,3307,4262,+,3307,protein_coding
MT-ND2,chrM,4470,5511,+,4470,protein_coding
MT-CO1,chrM,5904,7445,+,5904,protein_coding
MT-CO2,chrM,7586,8269,+,7586,protein_coding
MT-ATP8,chrM,8366,8572,+,8366,protein_coding


#### Mark Transcription Factors

deepSCENIC models gene regulation through transcription factors (TFs). We need to identify which genes in our dataset are TFs so the model knows which genes can act as regulators. This adds an `is_tf` column to `adata_rna.var` and stores the TF order in `adata_rna.uns['tf_order']`.

In [11]:
# Mark TFs in the RNA data
ds.pp.mark_tfs(adata_rna, tfs)
print(f"Marked {adata_rna.var['is_tf'].sum()} TFs in the dataset")

Marked 1463 TFs in the dataset


#### Add Gene Annotations

Gene annotations (chromosome positions) are needed for two purposes:
1. Computing the region-to-gene search space (which regions can regulate which genes)
2. Splitting features by chromosome for model evaluation

This adds `chromosome` and `tss` (transcription start site) columns to `adata_rna.var`.

In [12]:
# Add gene annotations (chromosome, TSS)
ds.pp.add_gene_annotation(adata_rna, annot)
adata_rna.var[["chromosome", "tss", "is_tf"]].head()

deepscenic.pp - INFO - Added ['chromosome', 'tss'] to 12109 / 15684 genes
deepscenic.pp - INFO -   (3575 genes not found in annotation)


,chromosome,tss,is_tf
A1BG,chr19,58353502,False
A1BG-AS1,<NA>,<NA>,False
A2M,chr12,9115919,False
A2M-AS1,<NA>,<NA>,False
A4GALT,chr22,42694986,False


In [20]:
# Remove genes without annotations
adata_rna = adata_rna[:, adata_rna.var[["chromosome", "tss", "is_tf"]].dropna().index]

### scATAC-seq Preprocessing

Load the chromatin accessibility matrix. This is required for training a deepSCENIC model, but not required for running inference or gene perturbation analyses using an already-trained model.

```{note}
We strongly recommend using **imputed** accessibility values (instead of raw) from [pycisTopic](https://pycistopic.readthedocs.io/en/latest/). The imputed matrix provides continuous accessibility values that are better suited for the VAE architecture than sparse binary fragment counts.
```

Imputed accessibility can be calculated on the fly by generating a mudata object using cell and region topic probabilities.

```python

cell_topic = sc.read_h5ad(<PATH_TO_CELL_TOPIC.h5ad>)
region_topic = sc.read_h5ad(<PATH_TO_REGION_TOPIC.h5ad>)

mdata = ds.pp.create_mudata(
    rna=adata_rna,
    cell_topic=cell_topic,
    region_topic=region_topic
)

```

By default the imputed values will be scaled by 10**6, this can be adapted
by setting `imputation_scaling_factor` in `ds.pp.create_mudata` 

The rest of the workflow is the same.

In [21]:
# Load scATAC-seq data (imputed accessibility from pycisTopic)
adata_atac = sc.read_h5ad(f"{data_dir}/imp_acc.h5ad")
print(f"ATAC data: {adata_atac.n_obs} cells x {adata_atac.n_vars} regions")

ATAC data: 936 cells x 288643 regions


#### Mark Differentially Accessible Regions (Optional)

Differentially Accessible Regions (DARs) are cell-type-specific regulatory regions identified through differential accessibility analysis. Marking DARs allows the model to upweight these biologically important regions during training.

```{note}
This step is **optional**. If you have DAR results from (for example) pycisTopic, you can load them here. Otherwise, skip this step.
```

In [22]:
# Optional: Mark DARs if you have pycisTopic differential accessibility results
# ds.pp.mark_dars(adata_atac, dar_dir="path/to/DARs/")

# Alternatively, provide a dictionary mapping cell types to region lists:
# dar_dict = {"CellType1": ["chr1:1000-2000", ...], "CellType2": [...]}
# ds.pp.mark_dars(adata_atac, dar_dict=dar_dict)

## Create MuData

deepSCENIC uses MuData to store multimodal data in a single object. The {func}`~deepscenic.pp.create_mudata` function combines the RNA and ATAC modalities and automatically parses region coordinates from the ATAC `var_names`. We expect the same cells in both data modalities. 

```{note}
The RNA and ATAC datasets must have identical cell barcodes (matching `obs_names`).
```

In [23]:
# Create MuData from RNA and ATAC
mdata = ds.pp.create_mudata(rna=adata_rna, atac=adata_atac)
mdata

MuData object with n_obs × n_vars = 936 × 300752
  2 modalities
    rna:	936 x 12109
      var:	'n_cells', 'is_tf', 'chromosome', 'tss'
      uns:	'log1p'
    atac:	936 x 288643
      var:	'chromosome', 'start', 'end'

### Add Cell Metadata (Optional)

If you have cell-level annotations (cell type, sample ID, batch, etc.), you can add them to `mdata.obs`. These annotations will be saved with the MuData and can be used later for visualization and analysis.

```{note}
Any columns you add to `mdata.obs` will be preserved when saving and loading the data. This is useful for annotating plots by cell type, grouping cells in analyses, or filtering by experimental conditions.
```

In [24]:
# Example: Load cell metadata from a CSV file
import pandas as pd

metadata = pd.read_csv(data_dir + "cells_metadata.csv", index_col=0)
print(f"Metadata columns: {list(metadata.columns)}")

# Add columns to MuData observations
mdata.obs["cell_line"] = mdata.obs_names.map(metadata["MMline"])
mdata.obs["cell_state"] = mdata.obs_names.map(metadata["lineState"])

# Verify the annotations were added
mdata.obs[["cell_line", "cell_state"]].head()

Metadata columns: ['MMline', 'lineState']


,cell_line,cell_state
MM001_0,MM001,MEL
MM001_1,MM001,MEL
MM001_2,MM001,MEL
MM001_3,MM001,MEL
MM001_4,MM001,MEL


## MuData Processing

Now we apply the preprocessing steps that operate on the combined MuData object.

### Compute Region-to-Gene Search Space

The region-to-gene (R2G) search space defines which genomic regions can potentially regulate which genes. This is based on genomic distance: regions closer to a gene's transcription start site (TSS) are more likely to regulate that gene.

The function computes a distance-weighted penalty matrix where:
- `max_distance`: Maximum distance from TSS to consider (default: 1Mb)
- `sigma`: Gaussian decay parameter (default: 100kb) - regions closer to TSS get lower penalty

In [25]:
# Compute r2g penalty matrix (distance-based)
ds.pp.compute_r2g_penalty(
    mdata,
    max_distance=1_000_000,  # 1Mb
    sigma=100_000,  # 100kb Gaussian sigma for decay penalty
)

Computing R2G matrix: 100%|██████████| 24/24 [00:05<00:00,  4.19it/s]
deepscenic.pp.search_space - INFO - Reordered RNA modality: 1446 TFs first, then 10663 other genes


### Split Data for Training and Evaluation

deepSCENIC uses two independent splits during main training:

1. **Cell split**: Cells are randomly assigned to training and test sets.
2. **Feature split**: Genes and ATAC regions are assigned by chromosome.

Genes are assigned using the chromosome containing their transcription start site (TSS), while ATAC regions are assigned using their genomic coordinates. In this tutorial, chromosome 16 is held out.

During main training, training cells are used for optimization and test cells for validation. RNA and ATAC reconstruction losses include only training-chromosome genes and regions, and tf2r is trained from training-region sequences. Held-out regions are evaluated separately to assess chromosome-level generalization and select the best checkpoint.

```{important}
The chromosome split is used only during main training. In the subsequent r2g finetuning step, tf2r is recomputed for every region, all genes contribute to RNA reconstruction, and all r2g links are optimized together. The cell train/test split is retained.
```

| Step | Optimization cells | Validation cells | Optimization regions | Validation regions | RNA loss | ATAC loss | Trainable components |
|:-----|:-------------------|:-----------------|:---------------------|:-------------------|:---------|:----------|:---------------------|
| **Phase 1:** Main training | Training cells | Test cells | Training regions | Training regions; held-out regions for chromosome evaluation | Training genes | Training regions | Complete model |
| **Phase 2:** r2g finetuning | Training cells | Test cells | All regions | All regions | All genes | None | All r2g links |

All annotated TFs remain available as encoder inputs in both phases because TF inputs are selected using `is_tf`, independently of the chromosome split.

In [27]:
# Split cells (80% train, 20% test)
ds.pp.split_cells(mdata, test_fraction=0.2, seed=42)
print(f"Cell split: {(mdata.obs['split'] == 'train').sum()} train, {(mdata.obs['split'] == 'test').sum()} test")

deepscenic.pp - INFO - Split cells: 748 train, 188 test


Cell split: 748 train, 188 test


In [28]:
# Split features by chromosome
ds.pp.split_features_by_chromosome(
    mdata,
    test_chromosomes=["chr16"],
)
print(
    f"Gene split: {(mdata['rna'].var['split'] == 'train').sum()} train, {(mdata['rna'].var['split'] == 'test').sum()} test"
)
print(
    f"Region split: {(mdata['atac'].var['split'] == 'train').sum()} train, {(mdata['atac'].var['split'] == 'test').sum()} test"
)

deepscenic.pp - INFO - Split features by chromosome: 8608 ATAC regions, 527 genes (63 TFs) in test set


Gene split: 11582 train, 527 test
Region split: 280035 train, 8608 test


## Saving and Loading

deepSCENIC stores all preprocessed data in a single `.h5mu` file. This includes both modalities, all annotations, the R2G matrix, and split information.

### Save Preprocessed Data

Use {func}`~deepscenic.write` to save the preprocessed MuData. The function validates the data against the deepSCENIC schema before saving to ensure all required fields are present.

In [31]:
# Save preprocessed MuData
ds.write(mdata, data_dir + "preprocessed_data.h5mu")

deepscenic.io - INFO - Wrote /data/groups/vib.ai/stein.aerts/gpartel/data/MM_lines_demo/preprocessed_data.h5mu: 936 cells


### Load Preprocessed Data

In future sessions, use {func}`~deepscenic.read` to load your preprocessed data. The function validates the schema on load to ensure data integrity.

In [26]:
# Load preprocessed MuData in future sessions
# mdata = ds.read("./data/MM_lines_results/preprocessed_data.h5mu")

## Next Steps

In the next tutorial, we'll train the deepSCENIC model:

- {doc}`02_training`